In [40]:
# pip install agentic-doc
# ZzZwNjBhYXQ2ZDlsMjgxNWpnaWZnOlpCMndrckVtQ0VUVEdHcXhlSUJ6M1RDblFKeWQ3Y1NG

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import json
import time
import logging
import pandas as pd
from openai import AzureOpenAI
import fitz  # PyMuPDF library
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# --- Configure Logging for Verbose Output ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# --- API Constants and Configuration ---
# Load from environment variables (set in .env file)
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
MODEL_NAME = os.getenv("AZURE_OPENAI_MODEL_NAME", "o3-mini-1")

# --- File and Batching Configuration ---
INPUT_PDF_PATH = 'Contract_voorbeeld.pdf' 

# --- UPDATED: Added a filename for the parsed text output ---
OUTPUT_TXT_PATH = 'parsed_contract_text.txt'
OUTPUT_EXCEL_PATH = 'TenderAnalyse_Resultaten.xlsx'
# --- END OF UPDATE ---

CHUNK_SIZE_IN_PAGES = 3
MAX_COMPLETION_TOKENS = 4000 

# --- Initialize Azure OpenAI Client ---
try:
    client = AzureOpenAI(
        azure_endpoint=AZURE_ENDPOINT,
        api_key=API_KEY,
        api_version=API_VERSION,
    )
    logging.info("Azure OpenAI client succesvol geïnitialiseerd.")
except Exception as e:
    logging.error(f"Initialisatie van Azure OpenAI client mislukt: {e}")
    client = None

In [42]:
def load_and_chunk_document(file_path: str, chunk_size: int) -> list:
    """
    UPDATED: Leest een PDF-bestand, extraheert de tekst, slaat de volledige tekst op
    in een .txt-bestand, en groepeert vervolgens de pagina's in chunks voor analyse.

    Args:
        file_path: Pad naar het .pdf-invoerbestand.
        chunk_size: Aantal pagina's per chunk.

    Returns:
        Een lijst van strings, waarbij elke string een chunk van meerdere pagina's tekst is.
    """
    if not os.path.exists(file_path):
        logging.error(f"Input PDF-bestand '{file_path}' niet gevonden.")
        return []

    logging.info(f"Start het extraheren van tekst uit PDF: {file_path}")
    doc = None
    try:
        doc = fitz.open(file_path)
        logging.info(f"{doc.page_count} pagina's gevonden in het document.")

        all_pages_text = []
        for page_num, page in enumerate(doc):
            text = page.get_text("text").strip()
            if text:
                all_pages_text.append(f"--- PAGINA {page_num + 1} ---\n{text}")

        if not all_pages_text:
            logging.error("Geen tekstinhoud gevonden in het PDF-bestand.")
            return []
        
        full_document_text = "\n\n".join(all_pages_text)
        logging.info(f"Tekst succesvol geëxtraheerd uit {len(all_pages_text)} pagina's.")

        # --- NEW: Logic to save the parsed text to a .txt file ---
        try:
            with open(OUTPUT_TXT_PATH, 'w', encoding='utf-8') as f:
                f.write(full_document_text)
            logging.info(f"De volledige geparste tekst is opgeslagen in: {OUTPUT_TXT_PATH}")
        except Exception as e:
            logging.warning(f"Kon het geparste .txt-bestand niet opslaan: {e}")
        # --- END OF NEW LOGIC ---

        # Groepeer de pagina's in chunks
        chunks = []
        for i in range(0, len(all_pages_text), chunk_size):
            chunk_content = "\n\n".join(all_pages_text[i:i + chunk_size])
            chunks.append(chunk_content)

        logging.info(f"Document opgedeeld in {len(chunks)} chunks van maximaal {chunk_size} pagina's.")
        return chunks

    except Exception as e:
        logging.error(f"Kon het PDF-bestand niet verwerken: {e}")
        return []
    finally:
        if doc:
            doc.close()

In [43]:
def analyze_chunk_for_questions(text_input: str) -> list:
    """
    Analyseert een tekst-input met de LLM om vragen voor de NvI te genereren.
    """
    if not client:
        logging.error("API client is niet geïnitialiseerd. Analyse wordt overgeslagen.")
        return []

    system_prompt = """
    U bent een zeer ervaren tender manager en contractspecialist werkzaam voor een groot Nederlands infrastructuur-bedrijf. U analyseert een concept 'Basisovereenkomst (UAV-GC 2005)' van TenneT voor het project "BRP station Nijverdal 110kV". Uw taak is om de tekst kritisch te beoordelen en potentiële onduidelijkheden te identificeren. Formuleer op basis hiervan duidelijke, professionele vragen voor de Nota van Inlichtingen (NvI).

    Focus op ambiguïteit, tegenstrijdigheden, ontbrekende informatie en risico's.

    VOORBEELD VAN EEN GOEDE VRAAG:
    - Document Referentie: VS1 Bijlage A-10
    - Geciteerde Tekst: "In de VS1 bijlage A10 'ombouwplan' zijn door de OIV'r en WV'r middels 'wolken' aangegeven opmerkingen toegevoegd. Mogelijk hebben deze opmerkingen ook tot gevolg dat VS1 Bijlage A09 'Voorlopig Ombouwplan en VNB-plan' eveneens aangepast te worden."
    - Vraag voor NvI: "Kandidaat dient gedurende de tenderfase de in de 'wolken' opgenomen opmerkingen te verwerken in een te herziene VS1 bijlage A10. Indien andere documenten als gevolg van het verwerken van deze aanpassingen ook gewijzigd moeten worden, dient de kandidaat deze door te geven en vervolgens aan te passen. Kunt u bevestigen dat onze interpretatie correct is?"

    Uw antwoord MOET een enkel, geldig JSON-object zijn met één sleutel, "vragen", die een lijst van woordenboeken bevat. Elk woordenboek moet de volgende sleutels hebben:
    - "document_referentie": Een korte, precieze verwijzing naar de locatie in het document (bijv., "Pagina 15, Artikel 16, lid 1").
    - "geciteerde_tekst": Het exacte tekstfragment dat tot de vraag leidt.
    - "vraag_voor_nvi": De helder geformuleerde, professionele vraag in het Nederlands.

    Genereer alleen vragen bij duidelijke onduidelijkheden. Lever geen tekst of uitleg buiten de vereiste JSON-structuur.
    """

    try:
        # --- LOGGING UPDATED FOR CLARITY ---
        logging.info(f"Volledige tekst (lengte: {len(text_input)} karakters) wordt naar LLM gestuurd voor NvI-vragen...")
        
        response = client.chat.completions.create(model=MODEL_NAME, messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": text_input}], max_completion_tokens=MAX_COMPLETION_TOKENS, response_format={"type": "json_object"})
        results_json = json.loads(response.choices[0].message.content)
        
        if "vragen" in results_json and isinstance(results_json["vragen"], list):
            logging.info(f"{len(results_json['vragen'])} vragen gegenereerd.")
            return results_json["vragen"]
        return []
    except Exception as e:
        logging.error(f"Fout tijdens het genereren van vragen: {e}")
        return []

In [44]:
import concurrent.futures
from tqdm import tqdm
import pandas as pd

def main():
    """
    Orchestreert het volledige dubbele analyse (NvI-vragen en risico's) en slaat de resultaten op in een Excel-bestand met twee tabbladen.
    """
    logging.info("--- START SCRIPT NVI-VRAGEN & RISICOANALYSE ---")

    # Stap 1: Laad de PDF, sla de tekst op, en verdeel in chunks
    document_chunks = load_and_chunk_document(INPUT_PDF_PATH, CHUNK_SIZE_IN_PAGES)
    if not document_chunks:
        logging.error("Geen tekst-chunks om te analyseren. Script stopt.")
        return

    # Stap 2: Verwerk de chunks parallel voor BEIDE analyses
    all_questions = []
    all_risks = []
    CONCURRENT_WORKERS = 10  # Aantal parallelle API-aanroepen

    logging.info(f"Start parallelle verwerking met {CONCURRENT_WORKERS} workers voor {len(document_chunks)} chunks.")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENT_WORKERS) as executor:
        # Dien voor elke chunk TWEE taken in: één voor vragen, één voor risico's
        future_to_task = {}
        for chunk in document_chunks:
            future_to_task[executor.submit(analyze_chunk_for_questions, chunk)] = 'vragen'
            future_to_task[executor.submit(analyze_chunk_for_risks, chunk)] = 'risicos'
        
        # Gebruik tqdm voor een voortgangsbalk die alle 40 taken volgt (20 chunks * 2 analyses)
        for future in tqdm(concurrent.futures.as_completed(future_to_task), total=len(future_to_task)):
            task_type = future_to_task[future]
            try:
                result = future.result()
                if result:
                    if task_type == 'vragen':
                        all_questions.extend(result)
                    elif task_type == 'risicos':
                        all_risks.extend(result)
            except Exception as exc:
                logging.error(f"Een taak van het type '{task_type}' heeft een exceptie veroorzaakt: {exc}")

    # Stap 3: Verwerk en sla de resultaten op in een Excel-bestand
    if not all_questions and not all_risks:
        logging.warning("Geen vragen of risico's gegenereerd door de LLM.")
        logging.info("--- SCRIPT VOLTOOID ---")
        return

    logging.info(f"Totaal {len(all_questions)} vragen en {len(all_risks)} risico's gegenereerd.")

    # Maak DataFrames voor beide resultaten
    questions_df = pd.DataFrame(all_questions)
    risks_df = pd.DataFrame(all_risks)

    # Bereken een 'Risk Score' en sorteer de risico's
    if not risks_df.empty:
        # Zorg ervoor dat de scorekolommen numeriek zijn, anders mislukt de berekening
        risks_df['risico_kans'] = pd.to_numeric(risks_df['risico_kans'], errors='coerce')
        risks_df['risico_impact'] = pd.to_numeric(risks_df['risico_impact'], errors='coerce')
        risks_df['risk_score'] = risks_df['risico_kans'] * risks_df['risico_impact']
        risks_df = risks_df.sort_values(by='risk_score', ascending=False)


    try:
        # Gebruik de OUTPUT_EXCEL_PATH variabele uit Cell 1
        with pd.ExcelWriter(OUTPUT_EXCEL_PATH, engine='openpyxl') as writer:
            # Schrijf de volledige lijst met vragen naar het eerste tabblad
            questions_df.to_excel(writer, sheet_name='NvI Vragen', index=False)
            
            # Filter de DataFrame tot de top 50 risico's
            top_50_risks_df = risks_df.head(50)
            # Schrijf de gefilterde DataFrame naar het tweede tabblad
            top_50_risks_df.to_excel(writer, sheet_name='Risicoanalyse Top 50', index=False)

        logging.info(f"Analyse voltooid. Rapport met twee tabbladen opgeslagen in: {OUTPUT_EXCEL_PATH}")
    except Exception as e:
        logging.error(f"Opslaan van Excel-bestand mislukt: {e}")

    logging.info("--- SCRIPT VOLTOOID ---")


# --- Voer het script uit ---
# Zorg ervoor dat de bibliotheek tqdm geïnstalleerd is voor de voortgangsbalk: pip install tqdm
if __name__ == "__main__":
    main()

2025-09-03 16:33:36 - INFO - --- START SCRIPT NVI-VRAGEN & RISICOANALYSE ---
2025-09-03 16:33:36 - INFO - Start het extraheren van tekst uit PDF: Contract_voorbeeld.pdf
2025-09-03 16:33:36 - INFO - 60 pagina's gevonden in het document.
2025-09-03 16:33:36 - INFO - Tekst succesvol geëxtraheerd uit 60 pagina's.
2025-09-03 16:33:36 - INFO - De volledige geparste tekst is opgeslagen in: parsed_contract_text.txt
2025-09-03 16:33:36 - INFO - Document opgedeeld in 20 chunks van maximaal 3 pagina's.
2025-09-03 16:33:36 - INFO - Start parallelle verwerking met 10 workers voor 20 chunks.
2025-09-03 16:33:36 - INFO - Volledige tekst (lengte: 4221 karakters) wordt naar LLM gestuurd voor NvI-vragen...
2025-09-03 16:33:36 - INFO - Tekst-chunk wordt naar LLM gestuurd voor risicoanalyse...
2025-09-03 16:33:36 - INFO - Volledige tekst (lengte: 9988 karakters) wordt naar LLM gestuurd voor NvI-vragen...
2025-09-03 16:33:36 - INFO - Tekst-chunk wordt naar LLM gestuurd voor risicoanalyse...
2025-09-03 16:3